# Chapter 8 &mdash; Ultimately Periodic Sets and String Lengths

**Concept 10 of the Chapter 8 decomposition:** *Ultimately Periodic Sets, and the Lengths of Strings in a Regular Language*

Beyond some bound $b$, membership repeats with period $p$ &mdash; and that is true of every regular language.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Ultimately-Periodic-Sets/Concept-Ultimately-Periodic-Sets.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A set $S\subseteq\mathbb{N}$ is **ultimately periodic** if there are $b$ (a bound) and
$p>0$ (a period) with

$$\forall n \ge b:\quad n\in S \iff n+p \in S.$$

**Theorem.** The set of *lengths* of strings in any regular language is ultimately
periodic.

The reason is the lasso of Chapter 4: past $|Q|$ symbols the machine must be in a
cycle, and going round the cycle adds a fixed number to the length without changing
acceptance. The period divides the cycle length.

This gives a **non-regularity test** independent of the Pumping Lemma: exhibit a
language whose length set is not ultimately periodic &mdash; $\{0^{n^2}\}$, say &mdash; and
you are done.

## 2. Definitions

### Length sets, and the ultimate-periodicity test

In [ ]:
def length_set(D, upto, sigma=None):
    # {n : the DFA accepts SOME string of length n}.  Computed by walking the
    # set of states reachable in exactly n steps -- enumerating 2^n strings
    # is hopeless past n = 20 or so.
    sigma = sigma or sorted(D["Sigma"])
    cur, out = {D["q0"]}, set()
    for n in range(upto + 1):
        if cur & D["F"]: out.add(n)
        cur = {step_dfa(D, q, a) for q in cur for a in sigma}
    return out

def find_period(S, upto, maxp=40):
    for b in range(upto // 2):
        for p in range(1, maxp):
            if all(((n in S) == (n + p in S)) for n in range(b, upto - p)):
                return b, p
    return None

### Some machines to test

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

## 3. Tests

A simple regular language: lengths divisible by 3.

In [ ]:
D = re_dfa("(000)*")
S = length_set(D, 40, ['0'])
print("length set :", sorted(S)[:14], "...")
b, p = find_period(S, 40)
print("ultimately periodic with bound %d, period %d" % (b, p))
assert p == 3

The stamp language $(1^3+1^5)^*$: bound $Fr+1 = 8$, period 1.

In [ ]:
D = re_dfa("(111+11111)*")
S = length_set(D, 60, ['1'])
print("payable lengths :", sorted(S)[:14], "...")
b, p = find_period(S, 60)
print("bound %d, period %d   (Fr(3,5) = %d)" % (b, p, 3*5-3-5))
assert p == 1 and b <= 8
assert all(n in S for n in range(8, 61))

Every regular language's length set is ultimately periodic.

In [ ]:
for r in ["(0+1)*1(0+1)(0+1)", "0*1*", "(01)*", "(0+1)(0+1)(0+1)"]:
    D = re_dfa(r)
    S = length_set(D, 24)
    bp = find_period(S, 24)
    print("%-22s lengths %-24s -> (b,p) = %s"
          % (r, sorted(S)[:8], bp))
    assert bp is not None

A **non**-regular language fails the test: $\{0^{n^2}\}$ has no period.

In [ ]:
squares = {n*n for n in range(30)}
print("squares :", sorted(squares)[:10], "...")
bad = find_period(squares, 200, maxp=60)
print("ultimately periodic? ", bad)
assert bad is None
print("\nGaps between squares grow without bound, so no fixed period can work.")
print("Hence {0^(n^2)} is NOT regular -- proved without the Pumping Lemma.")

## 4. Exercises


1. Find $b$ and $p$ for $(11+111)^*$ by hand, then check.
2. Is $\{0^{2^n}\}$ regular? Use this test.
3. Does ultimate periodicity of the length set **imply** regularity? Find a counterexample.

In [ ]:
# Your work for the exercises above.